pip install datasets pandas

In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


In [2]:
df = dataset["train"].to_pandas()

print(df.iloc[0])                          # one full row
print(df["intent"].nunique())              # expect 27
print(df["intent"].value_counts().head())  # rows per intent
print(df["flags"].value_counts().head(10)) # language flags
print(df["response"].str.contains(r"\{\{").mean()) 

flags                                                          B
instruction     question about cancelling order {{Order Number}}
category                                                   ORDER
intent                                              cancel_order
response       I've understood you have a question regarding ...
Name: 0, dtype: object
27
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
Name: count, dtype: int64
flags
BL      5212
BLQ     2467
BIL     2138
BLM     1297
BILQ    1057
BLQZ     970
BLZ      902
BKL      862
BLMQ     600
BEL      533
Name: count, dtype: int64
0.48399821375409346


In [3]:
print(df.shape)
print(df.head())

(26872, 5)
   flags                                        instruction category  \
0      B   question about cancelling order {{Order Number}}    ORDER   
1    BQZ  i have a question about cancelling oorder {{Or...    ORDER   
2   BLQZ    i need help cancelling puchase {{Order Number}}    ORDER   
3     BL         I need to cancel purchase {{Order Number}}    ORDER   
4  BCELN  I cannot afford this order, cancel purchase {{...    ORDER   

         intent                                           response  
0  cancel_order  I've understood you have a question regarding ...  
1  cancel_order  I've been informed that you have a question ab...  
2  cancel_order  I can sense that you're seeking assistance wit...  
3  cancel_order  I understood that you need assistance with can...  
4  cancel_order  I'm sensitive to the fact that you're facing f...  


In [4]:
print(df.info())
print(df.isnull().sum())

print(df["category"].value_counts())
print(df["intent"].value_counts())

print(df.sample(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB
None
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
CONTACT         1999
INVOICE         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent     

In [5]:
from pathlib import Path
Path("data/raw").mkdir(parents=True, exist_ok=True)
df.to_csv("data/raw/bitext.csv", index=False)

In [6]:
import pandas as pd
df = pd.read_csv("data/raw/bitext.csv", keep_default_na=False)

In [7]:
for ch in "BMLICNPQWKEZ":
    df[f"flag_{ch}"] = df["flags"].str.contains(ch)

In [8]:
import re
pat = re.compile(r"\{\{(.+?)\}\}")
from collections import Counter
c = Counter(m for t in df["instruction"].tolist() + df["response"].tolist() for m in pat.findall(t))
print(c.most_common(20))

[('Order Number', 8029), ('Account Type', 5440), ('Account Category', 3900), ('Online Order Interaction', 2699), ('Customer Support Phone Number', 2635), ('Website URL', 2534), ('Customer Support Hours', 2325), ('Invoice Number', 1521), ('Person Name', 1249), ('Refund Amount', 1187), ('Online Company Portal Info', 1055), ('Tracking Number', 896), ('Date Range', 894), ('Delivery City', 840), ('Currency Symbol', 745), ('Client Last Name', 712), ('Salutation', 706), ('Delivery Country', 681), ('Settings', 625), ('Profile', 490)]


In [9]:
from pathlib import Path
Path("data/processed").mkdir(parents=True, exist_ok=True)

In [10]:
df["ticket_id"] = range(len(df))

In [11]:
left_out = ["newsletter_subscription", "check_cancellation_fee", "switch_account", "review"]
kept = df[~df["intent"].isin(left_out)]
abstain_pool = df[df["intent"].isin(left_out)]

SEED = 42
test = kept.groupby("intent").sample(n=4, random_state=SEED)       
rest = kept.drop(test.index)
dev = rest.groupby("intent").sample(n=2, random_state=SEED)
past = rest.drop(dev.index)
abstain = abstain_pool.sample(n=30, random_state=SEED)

for name, part in [("past", past), ("dev", dev), ("test", test), ("abstain", abstain)]:
    part.to_csv(f"data/processed/{name}.csv", index=False)

In [12]:
for name, part in [
    ("past", past),
    ("dev", dev),
    ("test", test),
    ("abstain", abstain)
]:
    print(name, part.shape)

past (22788, 18)
dev (46, 18)
test (92, 18)
abstain (30, 18)


In [13]:
parts = {"past": past, "dev": dev, "test": test, "abstain": abstain}
ids = {n: set(p["ticket_id"]) for n, p in parts.items()}

# no ticket appears in two sets
names = list(ids)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        assert not (ids[names[i]] & ids[names[j]]), (names[i], names[j])

# left-out intents appear only in the abstain set
for n in ["past", "dev", "test"]:
    assert not parts[n]["intent"].isin(left_out).any(), n
assert set(abstain["intent"]) <= set(left_out)

print("split checks passed")

split checks passed


In [59]:
import random
import re
from collections import Counter
from pathlib import Path

import pandas as pd

PROCESSED = Path("data/processed")
PAT = re.compile(r"\{\{(.+?)\}\}")

SITE = "www.shopease.example"
PHONE = "+91 00000 00000"
EMAIL = "support@shopease.example"
HOURS = "Monday to Friday, 9 AM to 6 PM"

# Policy facts: the policy documents you write MUST state exactly these values.
POLICY = {
    "refund_time": "5 to 7 business days",
    "standard": "4 to 6 business days",
    "express": "1 to 2 business days",
    "min_days": "4",
    "max_days": "6",
    "cutoff": "12 PM",
    "guarantee": "30 days",
    "closure": "7 days",
}

EXACT = {
    "Live Chat Support": "live chat", "Live Chat": "live chat", "Live Chat Feature": "live chat",
    "Online Customer Support Channel": "live chat",
    "Online Company Portal Info": "the ShopEase customer portal",
    "Online Order Interaction": "My Orders",
    "Settings": "Account Settings", "Profile": "Profile",
    "Currency Symbol": "₹", "Profile Type": "Business",
}

PEOPLE = [("Priya", "Sharma", "Ms."), ("Rahul", "Verma", "Mr."), ("Anita", "Singh", "Ms."),
          ("Vikram", "Khan", "Mr."), ("Sara", "Brown", "Ms."), ("Daniel", "Lee", "Mr.")]
CITIES = ["Delhi", "Mumbai", "Pune", "Chennai", "Jaipur", "Kolkata"]
MONTHS = ["August", "September", "October"]

FALLBACKS = Counter()   # placeholders left as their own text (should be UI labels)


def make_ctx(ticket_id):
    r = random.Random(int(ticket_id))          # same ticket, same values, every run
    first, last, sal = r.choice(PEOPLE)
    day, month = r.randint(1, 14), r.choice(MONTHS)
    return {
        "first": first, "last": last, "sal": sal, "full": f"{first} {last}",
        "cities": r.sample(CITIES, 3), "month": month,
        "date": f"{day} {month}", "date_range": f"{day} to {day + 14} {month}",
        "amount": r.randint(100, 5000),
        "order": f"ORD-{r.randint(10000, 99999)}", "invoice": f"INV-{r.randint(10000, 99999)}",
        "tracking": f"TRK{r.randint(100000000, 999999999)}", "ref": f"REF-{r.randint(10000, 99999)}",
        "acct": "ShopEase",
    }


def fill(name, ctx, in_response, asked, literal):
    n = name.strip()
    if n in EXACT:
        return EXACT[n]

    def has(p):
        return re.search(p, n, re.I) is not None

    # contact details
    if has(r"url"):
        slug = ("/login" if has(r"login") else "/contact" if has(r"contact") else
                "/claims" if has(r"claims") else "/reset-password" if has(r"pin|key|password") else
                "/profile" if has(r"profile") else "/help" if has(r"help|support") else "")
        return SITE + slug
    if has(r"website"):
        return SITE
    if has(r"phone|hotline|toll.?free|helpline|contact number|support number"):
        return PHONE
    if has(r"email"):
        return EMAIL
    if has(r"support days"):
        return "Monday to Friday"
    if has(r"opening time|start time"):
        return "9 AM"
    if has(r"closing time|end time"):
        return "6 PM"
    if has(r"hours"):
        return HOURS

    # policy facts
    if has(r"cut.?off|same-day"):
        return POLICY["cutoff"]
    if has(r"pickup"):
        return "2 hours"
    if has(r"guarantee"):
        return POLICY["guarantee"]
    if has(r"closure.*timeframe"):
        return POLICY["closure"]
    if has(r"refund.*time|refund processing|^timeframe$"):
        return POLICY["refund_time"]
    if has(r"number of days"):
        return "30"
    if has(r"(delivery|shipping).*(time|days)|^eta$"):
        if has(r"\bmin\b"):
            return POLICY["min_days"]
        if has(r"\bmax\b"):
            return POLICY["max_days"]
        if has(r"express|expedited"):
            return POLICY["express"]
        return POLICY["standard"]

    # money, names, dates, places
    if has(r"money amount"):
        return f"₹{ctx['amount']}"
    if has(r"refund amount"):
        return str(ctx["amount"])
    if has(r"salutation"):
        return ctx["sal"]
    if has(r"first name"):
        return ctx["first"]
    if has(r"last name"):
        return ctx["last"]
    if has(r"(person|full|client|representative|account|invoice) name|client name"):
        return ctx["full"]
    if has(r"date range"):
        return ctx["date_range"]
    if has(r"^month$"):
        return ctx["month"]
    if has(r"^year$"):
        return "2026"
    if has(r"\bdate\b"):
        return ctx["date"]
    if has(r"\bcity\b|destination"):
        digits = re.findall(r"\d", n)
        return ctx["cities"][(int(digits[0]) - 1) % 3] if digits else ctx["cities"][0]
    if has(r"zip"):
        return "110001"
    if has(r"country"):
        return "India"
    if has(r"store address"):
        return "12 MG Road, Pune"
    if has(r"store location"):
        return "Pune"
    if has(r"retail stores"):
        return "our retail stores"

    # company and misc
    if has(r"support team"):
        return "ShopEase Support Team"
    if has(r"^company ?name$|business name|^company$|^online store$"):
        return "ShopEase"
    if has(r"carrier"):
        return "SwiftShip"
    if has(r"e-commerce platform|online marketplace"):
        return "an online marketplace"
    if has(r"social media"):
        return "our social media pages"
    if has(r"compensation type"):
        return "refund"
    if has(r"shipping method"):
        return "Standard Shipping"
    if has(r"contact method|^contact_method$"):
        return "email"
    if has(r"support channel"):
        return "live chat, email and phone"

    # PIN / access key / key variants all mean "password"
    if has(r"\b(pin|key)\b") and not has(r"keyword"):
        if has(r"forgot|lost|stolen"):
            return "Forgot Password"
        if has(r"reset|change|edit"):
            return "Reset Password"
        if has(r"recover|retriev|restor"):
            return "Password Recovery"
        if has(r"manage|setting"):
            return "Password Settings"
        return "Password"

    # IDs and numbers
    if has(r"number|\bid\b|identifier") and not has(r"number of"):
        kind = ("tracking" if has(r"tracking|shipment") else "invoice" if has(r"invoice")
                else "order" if has(r"order") else "ref")
        if in_response:
            if literal:
                return literal                  # reuse the number written in the question
            if n.lower() not in asked:
                return n.lower()                # the reply is asking for it, e.g. "order number"
        return ctx[kind]
    if has(r"account (type|category)"):
        return ctx["acct"]

    FALLBACKS[n] += 1
    return n     # UI labels such as 'Contact Us' stay as written
def tidy(text):
    text = re.sub(r"\{+([^{}]*)\}+", r"\1", text)                 # '{Cancel Order}}' -> 'Cancel Order'
    text = re.sub(r"\b(the|your|our)\s+the\b", r"\1", text, flags=re.I)   # 'your the X' -> 'your X'
    text = re.sub(r"""(['"])([^'"]+)\1\s+or\s+\1\2\1""", r"\1\2\1", text)  # 'X' or 'X' -> 'X'
    return text

def clean_row(row):
    ctx = make_ctx(row["ticket_id"])
    raw_q, raw_a = row["instruction"], row["response"]
    asked = {m.lower() for m in PAT.findall(raw_q)}
    m = re.search(r"#\s?([A-Za-z0-9-]*\d{3,})", PAT.sub("", raw_q))
    literal = m.group(1) if m else None

    row["instruction"] = tidy(PAT.sub(lambda x: fill(x.group(1), ctx, False, asked, literal), raw_q))
    row["response"] = tidy(PAT.sub(lambda x: fill(x.group(1), ctx, True, asked, literal), raw_a))
    return row


def main():
    for name in ["past", "dev", "test", "abstain"]:
        d = pd.read_csv(PROCESSED / f"{name}.csv", keep_default_na=False, encoding="utf-8")
        d = d.apply(clean_row, axis=1)
        d.to_csv(PROCESSED / f"{name}_clean.csv", index=False, encoding="utf-8")
        left = (d["instruction"].str.contains("{{", regex=False).sum()
                + d["response"].str.contains("{{", regex=False).sum())
        print(f"{name}: {len(d)} rows, leftover placeholders: {left}")
    print("Left as written (should be UI labels):", FALLBACKS.most_common(30))


main()

past: 22788 rows, leftover placeholders: 0
dev: 46 rows, leftover placeholders: 0
test: 92 rows, leftover placeholders: 0
abstain: 30 rows, leftover placeholders: 0
Left as written (should be UI labels): [('Forgot Password', 219), ('Order Status', 187), ('Order Tracking', 79), ('Account Recovery', 36), ('Security', 34), ('Cancel Purchase', 28), ('Profile Settings', 26), ('Account Recovery Page', 24), ('Purchase History', 14), ('Company Account', 13), ('Reset Password', 9), ('Track Order', 9), ('Password Reset Page', 6), ('Privacy', 5), ('Submit', 4), ('Password', 4), ('Upgrade Account', 4), ('Account Change', 4), ('Security & Privacy', 3), ('User Account Settings', 3), ('User Account Recovery', 3), ('Account Security', 3), ('Profile Recovery', 3), ('Purchase Details', 3), ('Cancellation Policy', 2), ('Refund Policy', 2), ('My Purchases', 2), ('Product Category', 2), ('Save', 2), ('Restore Password', 2)]


In [15]:
d = pd.read_csv(
    "data/processed/past_clean.csv",
    keep_default_na=False
)

sample = d.sample(n=10, random_state=42)

for _, row in sample.iterrows():
    
    print("\nCustomer:", row["instruction"])
    print("Response:", row["response"])
    print("Intent:", row["intent"])
    print("-" * 80)


Customer: I need assistance to download bill #00108
Response: Sure! I completely understand your need for assistance in downloading bill 00108. Let me quickly guide you through the process. To download your bill, please follow these steps:
1. Log in to your account on our website.
2. Go to the 'Account' section.
3. Navigate to the 'Billing' tab.
4. Look for the option to 'Download Invoice' and click on it.
5. Select the invoice number 00108 from the available list.
6. The invoice will be downloaded to your device in PDF format.
If you encounter any issues or have any further questions, please don't hesitate to reach out to me. I'm here to help!"
Intent: get_invoice
--------------------------------------------------------------------------------

Customer: wanna see how long refunds take help me
Response: For sure! I understand your need for clarity regarding the timeframe for refunds. The time it takes for refunds to be processed can vary depending on various factors, such as the paym

In [16]:


PROCESSED = Path("data/processed")

for name in ["past", "dev", "test", "abstain"]:
    raw = pd.read_csv(PROCESSED / f"{name}.csv", keep_default_na=False, encoding="utf-8")
    d = pd.read_csv(PROCESSED / f"{name}_clean.csv", keep_default_na=False, encoding="utf-8")
    text = d["instruction"] + " " + d["response"]

    # cleaning must not drop or reorder rows
    assert len(raw) == len(d), f"{name}: row count changed"
    assert list(raw["ticket_id"]) == list(d["ticket_id"]), f"{name}: ticket order changed"

    braces = text.str.contains(r"\{\{|\}\}").sum()
    empty = ((d["instruction"].str.strip() == "") | (d["response"].str.strip() == "")).sum()
    repeats = text.str.contains(r"\b(\w+)\s+\1\b", case=False, regex=True)

    print(f"\n{name}_clean.csv ({len(d)} rows)")
    print("  rows with leftover braces:", braces)
    print("  rows with an empty question or answer:", empty)
    print("  rows with a repeated word (e.g. 'days days'):", repeats.sum())
    for ex in d.loc[repeats, "response"].head(2):
        print("   example:", ex[:150])

C:\Users\User\AppData\Local\Temp\ipykernel_3392\2392232105.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  repeats = text.str.contains(r"\b(\w+)\s+\1\b", case=False, regex=True)



past_clean.csv (22788 rows)
  rows with leftover braces: 0
  rows with an empty question or answer: 0
  rows with a repeated word (e.g. 'days days'): 2646
   example: I can sense that you're seeking assistance with canceling your purchase with the purchase number ORD-89422. I apologize for any inconvenience caused, 
   example: I understood that you need assistance with canceling your purchase with the order number ORD-18588. We apologize for any inconvenience this may have c

dev_clean.csv (46 rows)
  rows with leftover braces: 0
  rows with an empty question or answer: 0
  rows with a repeated word (e.g. 'days days'): 4
   example: I get that you're seeking assistance with canceling purchase ORD-20607, and I'm here to guide you through the process. Let's take the necessary steps 
   example: Thank you for reaching out! I completely understand your need to speak with our customer assistance team. Our dedicated team is available during Monda

test_clean.csv (92 rows)
  rows with lefto

C:\Users\User\AppData\Local\Temp\ipykernel_3392\2392232105.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  repeats = text.str.contains(r"\b(\w+)\s+\1\b", case=False, regex=True)
C:\Users\User\AppData\Local\Temp\ipykernel_3392\2392232105.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  repeats = text.str.contains(r"\b(\w+)\s+\1\b", case=False, regex=True)
C:\Users\User\AppData\Local\Temp\ipykernel_3392\2392232105.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  repeats = text.str.contains(r"\b(\w+)\s+\1\b", case=False, regex=True)


In [17]:

pat = r"\b(\w+)\s+\1\b"
hits = d[d["response"].str.contains(pat, case=False, regex=True)]

words = hits["response"].str.extract(pat, flags=re.I)[0].str.lower()
print(words.value_counts().head(15))

for resp in hits["response"].head(6):
    m = re.search(pat, resp, flags=re.I)
    print("...", resp[max(0, m.start() - 40): m.end() + 40], "...")

Series([], Name: count, dtype: int64)


C:\Users\User\AppData\Local\Temp\ipykernel_3392\1610697825.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hits = d[d["response"].str.contains(pat, case=False, regex=True)]


In [18]:
import re

repeat_pat = re.compile(r"\b([A-Za-z]+)\s+\1\b", re.IGNORECASE)
repeats = text.apply(lambda t: bool(repeat_pat.search(t)))

In [19]:
%run scripts/load_data.py

abstain 30
dev 46
past 22788
test 92
documents: 18
intent mappings: 23


In [20]:
from db import get_connection
conn = get_connection()
q = """SELECT t.intent, d.title FROM tickets t
       JOIN intent_docs i ON i.intent = t.intent
       JOIN documents d ON d.doc_id = i.doc_id
       WHERE t.split = 'test' LIMIT 5"""
for r in conn.execute(q):
    print(dict(r))
print(conn.execute("SELECT COUNT(*) FROM tickets WHERE split='abstain' AND intent IN (SELECT intent FROM intent_docs)").fetchone()[0])

{'intent': 'cancel_order', 'title': 'Cancelling an order'}
{'intent': 'cancel_order', 'title': 'Cancelling an order'}
{'intent': 'cancel_order', 'title': 'Cancelling an order'}
{'intent': 'cancel_order', 'title': 'Cancelling an order'}
{'intent': 'change_order', 'title': 'Changing an order'}
0


pip install rank_bm25

In [21]:
import sys
sys.path.insert(0, "src")
from collections import defaultdict
from db import get_connection
from retrieval import BM25Docs

conn = get_connection()
searcher = BM25Docs(conn)

gold = defaultdict(set)                       # intent -> correct document ids
for r in conn.execute("SELECT intent, doc_id FROM intent_docs"):
    gold[r["intent"]].add(r["doc_id"])


def evaluate(rows, searcher):
    hits = {1: 0, 3: 0, 5: 0}
    rr = 0.0
    for r in rows:
        ranked = [d for d, _ in searcher.search(r["query"], k=18)]
        ranks = [i + 1 for i, d in enumerate(ranked) if d in gold[r["intent"]]]
        first = min(ranks)
        for k in hits:
            hits[k] += first <= k
        rr += 1 / first
    n = len(rows)
    return {"n": n, **{f"hit@{k}": round(v / n, 3) for k, v in hits.items()}, "MRR": round(rr / n, 3)}


dev = conn.execute("SELECT intent, flags, query FROM tickets WHERE split='dev'").fetchall()
print("dev:", evaluate(dev, searcher))

W1116 11:17:34.226000 3392 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


dev: {'n': 46, 'hit@1': 0.348, 'hit@3': 0.587, 'hit@5': 0.696, 'MRR': 0.508}


In [22]:
def failures(rows, searcher, show=8):
    shown = 0
    for r in rows:
        ranked = [d for d, _ in searcher.search(r["query"], k=3)]
        if ranked[0] not in gold[r["intent"]]:
            print(f'[{r["intent"]}] {r["query"]}')
            print(f'   gold: {sorted(gold[r["intent"]])}')
            print(f'   top3: {ranked}\n')
            shown += 1
            if shown == show:
                break

failures(dev, searcher)

[change_order] modifying purchase ORD-76074
   gold: ['change-order']
   top3: ['cancel-order', 'change-order', 'complaints']

[change_order] i cant correct purchase ORD-49437
   gold: ['change-order']
   top3: ['invoices', 'complaints', 'shipping-address']

[change_shipping_address] I want to know about correcting the address
   gold: ['shipping-address']
   top3: ['invoices', 'shipping-address', 'create-account']

[change_shipping_address] want assistance to update the address
   gold: ['shipping-address']
   top3: ['edit-account', 'shipping-address', 'track-order']

[check_invoice] could I see the billfrom Sara Brown?
   gold: ['invoices']
   top3: ['delivery-options', 'place-order', 'cancel-order']

[check_payment_methods] how to list your allowed payment methods?
   gold: ['payment-methods']
   top3: ['change-order', 'invoices', 'payment-methods']

[check_payment_methods] can uhelp me check the bloody allowed paymentmethods
   gold: ['payment-methods']
   top3: ['invoices', 'refun

In [23]:
tune = conn.execute(
    "SELECT intent, flags, query FROM tickets WHERE split='past' AND ticket_id % 20 = 0"
).fetchall()                                   # about 1,100 tickets, always the same ones
print("tuning sample:", evaluate(tune, searcher))

for letter in "ZKQ":                           # typos, keyword-style, colloquial
    subset = [r for r in tune if letter in r["flags"]]
    print(letter, evaluate(subset, searcher))

tuning sample: {'n': 1137, 'hit@1': 0.387, 'hit@3': 0.616, 'hit@5': 0.749, 'MRR': 0.543}
Z {'n': 225, 'hit@1': 0.364, 'hit@3': 0.542, 'hit@5': 0.698, 'MRR': 0.508}
K {'n': 92, 'hit@1': 0.402, 'hit@3': 0.609, 'hit@5': 0.696, 'MRR': 0.548}
Q {'n': 370, 'hit@1': 0.368, 'hit@3': 0.573, 'hit@5': 0.73, 'MRR': 0.52}


In [24]:
by_intent = defaultdict(list)
for r in tune:
    top = searcher.search(r["query"], k=1)[0][0]
    by_intent[r["intent"]].append(top in gold[r["intent"]])

for i, v in sorted(by_intent.items(), key=lambda x: sum(x[1]) / len(x[1])):
    print(f"{i:28s} {sum(v)/len(v):.2f}  (n={len(v)})")

place_order                  0.02  (n=49)
registration_problems        0.06  (n=48)
delivery_period              0.10  (n=50)
track_order                  0.10  (n=50)
get_refund                   0.12  (n=50)
change_order                 0.14  (n=50)
change_shipping_address      0.15  (n=48)
set_up_shipping_address      0.16  (n=49)
track_refund                 0.18  (n=50)
payment_issue                0.22  (n=50)
check_refund_policy          0.28  (n=50)
recover_password             0.34  (n=50)
delivery_options             0.35  (n=49)
check_payment_methods        0.38  (n=48)
edit_account                 0.44  (n=50)
delete_account               0.51  (n=49)
create_account               0.53  (n=49)
complaint                    0.63  (n=49)
contact_customer_service     0.72  (n=50)
contact_human_agent          0.82  (n=50)
cancel_order                 0.84  (n=50)
get_invoice                  0.84  (n=50)
check_invoice                0.96  (n=49)


In [25]:
import importlib, retrieval
importlib.reload(retrieval)
from retrieval import BM25Docs, tokenize_raw, tokenize_clean

In [26]:
for name, tok in [("raw", tokenize_raw), ("clean", tokenize_clean)]:
    s = BM25Docs(conn, tokenize=tok)
    print(name, "dev:   ", evaluate(dev, s))
    print(name, "tuning:", evaluate(tune, s))
    for letter in "ZKQ":
        print("   ", letter, evaluate([r for r in tune if letter in r["flags"]], s))

raw dev:    {'n': 46, 'hit@1': 0.326, 'hit@3': 0.543, 'hit@5': 0.63, 'MRR': 0.475}
raw tuning: {'n': 1137, 'hit@1': 0.296, 'hit@3': 0.476, 'hit@5': 0.594, 'MRR': 0.443}
    Z {'n': 225, 'hit@1': 0.267, 'hit@3': 0.431, 'hit@5': 0.556, 'MRR': 0.418}
    K {'n': 92, 'hit@1': 0.337, 'hit@3': 0.489, 'hit@5': 0.587, 'MRR': 0.464}
    Q {'n': 370, 'hit@1': 0.27, 'hit@3': 0.449, 'hit@5': 0.573, 'MRR': 0.421}
clean dev:    {'n': 46, 'hit@1': 0.348, 'hit@3': 0.587, 'hit@5': 0.696, 'MRR': 0.508}
clean tuning: {'n': 1137, 'hit@1': 0.387, 'hit@3': 0.616, 'hit@5': 0.749, 'MRR': 0.543}
    Z {'n': 225, 'hit@1': 0.364, 'hit@3': 0.542, 'hit@5': 0.698, 'MRR': 0.508}
    K {'n': 92, 'hit@1': 0.402, 'hit@3': 0.609, 'hit@5': 0.696, 'MRR': 0.548}
    Q {'n': 370, 'hit@1': 0.368, 'hit@3': 0.573, 'hit@5': 0.73, 'MRR': 0.52}


In [27]:
import importlib, retrieval
importlib.reload(retrieval)
from retrieval import BM25Docs, DenseDocs, HybridDocs

bm25 = BM25Docs(conn)
dense = DenseDocs(conn)
hybrid = HybridDocs(bm25, dense)

for name, s in [("bm25_clean", bm25), ("dense", dense), ("hybrid_rrf", hybrid)]:
    print(name, "tuning:", evaluate(tune, s))
    for letter in "ZKQ":
        print("   ", letter, evaluate([r for r in tune if letter in r["flags"]], s))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

bm25_clean tuning: {'n': 1137, 'hit@1': 0.387, 'hit@3': 0.616, 'hit@5': 0.749, 'MRR': 0.543}
    Z {'n': 225, 'hit@1': 0.364, 'hit@3': 0.542, 'hit@5': 0.698, 'MRR': 0.508}
    K {'n': 92, 'hit@1': 0.402, 'hit@3': 0.609, 'hit@5': 0.696, 'MRR': 0.548}
    Q {'n': 370, 'hit@1': 0.368, 'hit@3': 0.573, 'hit@5': 0.73, 'MRR': 0.52}
dense tuning: {'n': 1137, 'hit@1': 0.724, 'hit@3': 0.882, 'hit@5': 0.951, 'MRR': 0.817}
    Z {'n': 225, 'hit@1': 0.622, 'hit@3': 0.831, 'hit@5': 0.933, 'MRR': 0.746}
    K {'n': 92, 'hit@1': 0.772, 'hit@3': 0.902, 'hit@5': 0.957, 'MRR': 0.849}
    Q {'n': 370, 'hit@1': 0.692, 'hit@3': 0.878, 'hit@5': 0.951, 'MRR': 0.799}
hybrid_rrf tuning: {'n': 1137, 'hit@1': 0.549, 'hit@3': 0.814, 'hit@5': 0.885, 'MRR': 0.697}
    Z {'n': 225, 'hit@1': 0.467, 'hit@3': 0.809, 'hit@5': 0.858, 'MRR': 0.647}
    K {'n': 92, 'hit@1': 0.543, 'hit@3': 0.826, 'hit@5': 0.891, 'MRR': 0.693}
    Q {'n': 370, 'hit@1': 0.527, 'hit@3': 0.814, 'hit@5': 0.865, 'MRR': 0.681}


%pip install -U transformers sentence-transformers

In [28]:
import csv, os
from datetime import date

RESULTS = "results/retrieval_results.csv"

def log(system, sample, metrics, note=""):
    os.makedirs("results", exist_ok=True)
    new_file = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(["date", "system", "sample", "n", "hit@1", "hit@3", "hit@5", "MRR", "note"])
        w.writerow([date.today().isoformat(), system, sample,
                    metrics["n"], metrics["hit@1"], metrics["hit@3"], metrics["hit@5"],
                    metrics["MRR"], note])

In [29]:
log("bm25_raw",   "tuning", evaluate(tune, BM25Docs(conn, tokenize=tokenize_raw)), "no stemming, no stopwords")
log("bm25_clean", "tuning", evaluate(tune, bm25), "stemming, stopwords, order numbers removed")
log("dense",      "tuning", evaluate(tune, dense), "bge-small-en-v1.5")
log("hybrid_rrf", "tuning", evaluate(tune, hybrid), "equal-weight RRF")

In [30]:
for letter in "ZKQ":
    subset = [r for r in tune if letter in r["flags"]]
    log("dense", f"tuning_flag_{letter}", evaluate(subset, dense))

In [31]:
from collections import Counter
pairs = Counter()
for r in tune:
    top = dense.search(r["query"], k=1)[0][0]
    if top not in gold[r["intent"]]:
        pairs[(r["intent"], top)] += 1
for (intent, wrong), n in pairs.most_common(12):
    print(f"{n:3d}  {intent:26s} -> {wrong}")

 33  delivery_period            -> track-order
 18  check_refund_policy        -> refund-tracking
 18  registration_problems      -> password-recovery
 17  get_refund                 -> payment-problems
 16  delivery_options           -> track-order
 14  place_order                -> contact-support
 13  track_order                -> refund-tracking
  9  delivery_options           -> payment-methods
  8  get_refund                 -> complaints
  8  registration_problems      -> complaints
  8  track_refund               -> payment-problems
  7  get_refund                 -> payment-methods


In [32]:
rows_a = tune
rows_b = conn.execute(
    "SELECT intent, flags, query FROM tickets WHERE split='past' AND ticket_id % 20 = 10").fetchall()

def precompute(rows):
    return [([d for d, _ in dense.search(r["query"], k=18)],
             [d for d, _ in bm25.search(r["query"], k=18)]) for r in rows]

def eval_weight(rows, pre, w, c=60):
    h1 = h5 = mrr = 0
    for r, (dr, br) in zip(rows, pre):
        s = {}
        for rank, d in enumerate(dr, 1): s[d] = s.get(d, 0) + w / (c + rank)
        for rank, d in enumerate(br, 1): s[d] = s.get(d, 0) + (1 - w) / (c + rank)
        ranked = sorted(s, key=s.get, reverse=True)
        first = min(i + 1 for i, d in enumerate(ranked) if d in gold[r["intent"]])
        h1 += first == 1; h5 += first <= 5; mrr += 1 / first
    n = len(rows)
    return round(h1 / n, 3), round(h5 / n, 3), round(mrr / n, 3)

pre_a, pre_b = precompute(rows_a), precompute(rows_b)
for w in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(w, "tune:", eval_weight(rows_a, pre_a, w), " confirm:", eval_weight(rows_b, pre_b, w))

0.5 tune: (0.595, 0.887, 0.721)  confirm: (0.583, 0.87, 0.703)
0.6 tune: (0.611, 0.916, 0.74)  confirm: (0.591, 0.901, 0.719)
0.7 tune: (0.658, 0.934, 0.774)  confirm: (0.629, 0.928, 0.752)
0.8 tune: (0.706, 0.948, 0.805)  confirm: (0.675, 0.943, 0.786)
0.9 tune: (0.72, 0.953, 0.815)  confirm: (0.7, 0.942, 0.802)
1.0 tune: (0.724, 0.951, 0.817)  confirm: (0.711, 0.948, 0.807)


In [33]:
shown = Counter()
targets = [("check_refund_policy", "refund-tracking"), ("track_order", "refund-tracking"),
           ("place_order", "contact-support"), ("delivery_options", "track-order"),
           ("get_refund", "payment-problems"), ("registration_problems", "password-recovery")]
for r in tune:
    top = dense.search(r["query"], k=1)[0][0]
    key = (r["intent"], top)
    if key in targets and shown[key] < 3:
        shown[key] += 1
        print(f'{r["intent"]} -> {top}\n   {r["query"]}')

check_refund_policy -> refund-tracking
   can I check in what cases can I ask to be reimbursed?
check_refund_policy -> refund-tracking
   want assistance to check how long refunds take
check_refund_policy -> refund-tracking
   can you help me see how long refunds take?
delivery_options -> track-order
   where do I see the shipment methods?
delivery_options -> track-order
   I want to see the delivery methods, help me
delivery_options -> track-order
   I don't know what to do to see the delivery options
get_refund -> payment-problems
   I want assistace to request a compensation of my money
get_refund -> payment-problems
   assistance requesting a compensation of money
get_refund -> payment-problems
   help demanding rebates of money
place_order -> contact-support
   help em buy several of ur article
place_order -> contact-support
   assistance to buiy some of your product
place_order -> contact-support
   assistance shopping some of your product
registration_problems -> password-recove

In [34]:
import importlib, retrieval
importlib.reload(retrieval)
from retrieval import BM25Docs, DenseDocs, DenseChunks
chunked = DenseChunks(conn)
print("chunks:", len(chunked.chunks))
for name, s in [("dense_doc", dense), ("dense_chunks", chunked)]:
    print(name, "tune:   ", evaluate(tune, s))
    print(name, "confirm:", evaluate(rows_b, s))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

chunks: 67
dense_doc tune:    {'n': 1137, 'hit@1': 0.724, 'hit@3': 0.882, 'hit@5': 0.951, 'MRR': 0.817}
dense_doc confirm: {'n': 1140, 'hit@1': 0.711, 'hit@3': 0.871, 'hit@5': 0.948, 'MRR': 0.807}
dense_chunks tune:    {'n': 1137, 'hit@1': 0.73, 'hit@3': 0.916, 'hit@5': 0.97, 'MRR': 0.832}
dense_chunks confirm: {'n': 1140, 'hit@1': 0.744, 'hit@3': 0.918, 'hit@5': 0.969, 'MRR': 0.84}


In [35]:
log("dense_doc",    "tuning",  evaluate(tune, dense),    "bge-small, whole documents")
log("dense_chunks", "tuning",  evaluate(tune, chunked),  "bge-small, 67 chunks, max-chunk score")
log("dense_doc",    "confirm", evaluate(rows_b, dense),  "")
log("dense_chunks", "confirm", evaluate(rows_b, chunked), "")

In [36]:
import numpy as np

def top_scores(rows, s):
    return np.array([s.search(r["query"], k=1)[0][1] for r in rows])

abstain_rows = conn.execute(
    "SELECT intent, flags, query FROM tickets WHERE split='abstain'").fetchall()

ans = top_scores(tune, chunked)
abs_ = top_scores(abstain_rows, chunked)

print("answerable score percentiles 5/25/50/75:", np.percentile(ans, [5, 25, 50, 75]).round(3))
print("abstain score percentiles 25/50/75/95:  ", np.percentile(abs_, [25, 50, 75, 95]).round(3))
for p in [5, 10, 20]:
    t = np.percentile(ans, p)
    print(f"threshold = answerable p{p} ({t:.3f}): "
          f"answerable kept {np.mean(ans >= t):.0%}, abstain refused {np.mean(abs_ < t):.0%}")

answerable score percentiles 5/25/50/75: [0.58  0.647 0.689 0.729]
abstain score percentiles 25/50/75/95:   [0.591 0.621 0.655 0.695]
threshold = answerable p5 (0.580): answerable kept 95%, abstain refused 13%
threshold = answerable p10 (0.601): answerable kept 90%, abstain refused 33%
threshold = answerable p20 (0.636): answerable kept 80%, abstain refused 63%


In [37]:
abstain_rows = conn.execute(
    "SELECT ticket_id, intent, flags, query FROM tickets WHERE split='abstain'").fetchall()
abstain_dev   = [r for r in abstain_rows if r["ticket_id"] % 3 == 0]    # about 10: prompt development
abstain_final = [r for r in abstain_rows if r["ticket_id"] % 3 != 0]    # about 20: final evaluation only
print(len(abstain_dev), len(abstain_final))

7 23


In [38]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [39]:
import os
from dotenv import load_dotenv, dotenv_values
load_dotenv(override=True)
print("same as .env file:", os.environ.get("GEMINI_API_KEY") == dotenv_values(".env").get("GEMINI_API_KEY"))

same as .env file: True


In [40]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

for m in client.models.list():
    actions = getattr(m, "supported_actions", None)
    print(m.name, "|", actions)

models/gemini-2.5-flash | ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro | ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemma-4-26b-a4b-it | ['generateContent', 'countTokens']
models/gemma-4-31b-it | ['generateContent', 'countTokens']
models/gemini-2.5-flash-lite | ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-image | ['generateContent', 'countTokens', 'batchGenerateContent']
models/gemini-3.1-flash-lite | ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-3-pro-image | ['generateContent', 'countTokens', 'batchGenerateContent']
models/gemini-3.1-flash-image | ['generateContent', 'countTokens', 'batchGenerateContent']
models/gemini-3.1-flash-lite-image | ['generateContent', 'countTokens', 'batchGenerateContent']
models/gemini-nano-banana-2.1 | ['generateContent', 'countTokens', 'batch

In [41]:
import os
from dotenv import load_dotenv, dotenv_values

file_key = dotenv_values(".env").get("GEMINI_API_KEY")
env_key = os.environ.get("GEMINI_API_KEY")

print("same as .env file:", env_key == file_key)
print("length in .env:", len(file_key or ""))
print("spaces or quotes present:", bool(file_key) and (file_key != file_key.strip() or "'" in file_key or '"' in file_key))
print("GOOGLE_API_KEY also set:", "GOOGLE_API_KEY" in os.environ)

same as .env file: True
length in .env: 53
spaces or quotes present: False
GOOGLE_API_KEY also set: False


In [42]:
from dotenv import dotenv_values
k = dotenv_values(".env").get("GEMINI_API_KEY") or ""
print("length:", len(k), "| starts with:", k[:4])

length: 53 | starts with: AQ.A


In [43]:
for m in client.models.list():
    n = m.name.replace("models/", "")
    if "flash" in n and not any(x in n for x in ["image", "tts", "live", "audio", "embedding", "omni", "preview", "exp"]):
        print(n)

gemini-2.5-flash
gemini-2.5-flash-lite
gemini-3.1-flash-lite
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash


In [44]:
for name in ["gemini-3.5-flash-lite", "gemini-3.5-flash", "gemini-3.6-flash","gemini-2.5-flash","gemini-3.8-flash","gemini-2.5-flash-lite","gemini-3.1-flash-lite","gemini-3.7-flash"]:
    try:
        r = client.models.generate_content(model=name, contents="Reply with the single word: ok")
        print(name, "->", (r.text or "").strip())
    except Exception as e:
        print(name, "-> ERROR", getattr(e, "code", None), str(e)[:150])

gemini-3.5-flash-lite -> ok
gemini-3.5-flash -> ok
gemini-3.6-flash -> ok
gemini-2.5-flash -> ERROR 404 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to 
gemini-3.8-flash -> ok
gemini-2.5-flash-lite -> ERROR 404 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash-lite is no longer available to new users. Please update your cod
gemini-3.1-flash-lite -> ok
gemini-3.7-flash -> ERROR 503 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please 


In [45]:
from dotenv import load_dotenv
load_dotenv(override=True)
import importlib, llm
importlib.reload(llm)
from llm import LLM

gem = LLM(min_interval=6.5)          # replace with your own value from step 2
print(gem.generate("Say hello in five words."))
print(gem.generate("Say hello in five words."))
print("model:", gem.model, "| real calls:", gem.calls, "| cache hits:", gem.cache_hits)

Hello, how are you today?
Hello, how are you today?
model: gemini-3.5-flash-lite | real calls: 0 | cache hits: 2


In [46]:
import sys; sys.path.insert(0, "src")
import importlib, drafting
importlib.reload(drafting)
from drafting import draft
from llm import LLM

gem = LLM(min_interval=4.5)

dev10 = conn.execute("SELECT ticket_id, intent, query FROM tickets WHERE split='dev' ORDER BY ticket_id LIMIT 10").fetchall()
for r in dev10[:4]:
    out = draft(gem, chunked, r["query"])
    print(r["intent"], "|", r["query"])
    print("   ", out["draft"] or "(no draft: " + out["reason"] + ")", "\n")

cancel_order | I don't want purchase ORD-87590
    Hello! You can cancel an order at any time before it is dispatched [2]. To cancel online, sign in to the ShopEase customer portal, open **My Orders**, select the order, click **Cancel Order**, choose a reason, and confirm [2]. If you cannot cancel online, please contact customer support on +91 00000 00000 or support@shopease.example (Monday to Friday, 9 AM to 6 PM) and quote your order number [4]. 

cancel_order | want help canceling purchase ORD-20607
    You can cancel an order at any time before it is dispatched [2]. To cancel, sign in to the ShopEase customer portal and open **My Orders**, select the order, click **Cancel Order**, choose a reason, and confirm [2]. If you cannot cancel online, please contact customer support at +91 00000 00000 or support@shopease.example (Monday to Friday, 9 AM to 6 PM) and quote your order number ORD-20607 [1]. 

change_order | modifying purchase ORD-76074
    Hello! 

You can change an order until

In [47]:
abstain_dev = conn.execute(
    "SELECT ticket_id, intent, query FROM tickets WHERE split='abstain' AND ticket_id % 3 = 0").fetchall()

def abstain_rate(rows):
    outs = [draft(gem, chunked, r["query"]) for r in rows]
    return sum(o["abstained"] for o in outs), len(outs)

print("answerable dev tickets refused:", abstain_rate(dev10))
print("abstain tickets refused:       ", abstain_rate(abstain_dev))

answerable dev tickets refused: (3, 10)
abstain tickets refused:        (7, 7)


In [48]:
import importlib, drafting
importlib.reload(drafting)
from drafting import draft          # re-import AFTER the reload, so the name points to the new function

import inspect
print(inspect.signature(draft))

(llm, retriever, query, k=4, floor=0.58, system="You are a support assistant for ShopEase, an online store. You write draft replies that a human support agent will review before sending.\n\nRules:\n1. Use ONLY the numbered passages provided. Do not use outside knowledge and do not guess.\n2. Every sentence that states a fact (steps, time limits, contact details, policies) must end with the number of the passage that supports it, like [1] or [2][3].\n3. If the passages do not contain the information needed to answer the customer's question, reply with exactly: NOT_ENOUGH_EVIDENCE\n4. Be polite and concise, at most 120 words. Do not invent order details, dates or amounts.")


In [49]:
from collections import defaultdict
gold = defaultdict(set)
for r in conn.execute("SELECT intent, doc_id FROM intent_docs"):
    gold[r["intent"]].add(r["doc_id"])

dev_all = conn.execute("SELECT ticket_id, intent, query FROM tickets WHERE split='dev'").fetchall()
abstain_dev = conn.execute(
    "SELECT ticket_id, intent, query FROM tickets WHERE split='abstain' AND ticket_id % 3 = 0").fetchall()

def run_drafts(rows, system=drafting.SYSTEM):
    return [(r, draft(gem, chunked, r["query"], system=system)) for r in rows]

def summarize(results):
    n = len(results)
    refused = sum(o["abstained"] for _, o in results)
    answered = [(r, o) for r, o in results if not o["abstained"]]
    cited = sum(bool(o["cited"]) for _, o in answered)
    gold_cited = sum(any(o["passages"][i - 1][0] in gold[r["intent"]] for i in o["cited"])
                     for r, o in answered)
    print(f"n={n} | refused {refused} ({refused / n:.0%}) | of answered: "
          f"with citation {cited}/{len(answered)}, cites correct document {gold_cited}/{len(answered)}")

res_dev = run_drafts(dev_all)
res_abs = run_drafts(abstain_dev)
print("answerable dev:"); summarize(res_dev)
print("abstain dev:  "); summarize(res_abs)

answerable dev:
n=46 | refused 12 (26%) | of answered: with citation 34/34, cites correct document 31/34
abstain dev:  
n=7 | refused 7 (100%) | of answered: with citation 0/0, cites correct document 0/0


In [50]:
for r, o in res_dev:
    if o["abstained"]:
        print(r["intent"], "|", r["query"], "| top score:", round(o["passages"][0][2], 3), "| reason:", o["reason"])
        for d, t, s in o["passages"][:2]:
            print("     ", d, round(s, 3))

change_order | i cant correct purchase ORD-49437 | top score: 0.672 | reason: model_judgement
      invoices 0.672
      change-order 0.668
check_invoice | could I see the billfrom Sara Brown? | top score: 0.423 | reason: low_retrieval_score
      payment-problems 0.423
      edit-account 0.419
check_invoice | is it possible to locate the bill #37777? | top score: 0.657 | reason: model_judgement
      invoices 0.657
      invoices 0.607
check_refund_policy | I need to see in which cases can I ask for a reimbursement | top score: 0.674 | reason: model_judgement
      complaints 0.674
      invoices 0.625
create_account | where to create a Premium account? | top score: 0.631 | reason: model_judgement
      create-account 0.631
      create-account 0.596
delete_account | clsoe Basic account | top score: 0.541 | reason: low_retrieval_score
      create-account 0.541
      create-account 0.536
edit_account | correct information on Basic account | top score: 0.627 | reason: model_judgement
 

In [51]:
SYSTEM_B = drafting.SYSTEM.replace(
    "3. If the passages do not contain the information needed to answer the customer's question, reply with exactly: NOT_ENOUGH_EVIDENCE",
    "3. If the passages are about a different topic from the customer's request, reply with exactly: NOT_ENOUGH_EVIDENCE. "
    "If the request is short or unclear but the passages cover its most likely meaning, answer for that meaning. "
    "If the passages cover only part of the request, answer that part and say that customer support can help with the rest. "
    "Never fill any gap with outside knowledge."
)
assert SYSTEM_B != drafting.SYSTEM            # makes sure the replacement matched

res_dev_b = run_drafts(dev_all, system=SYSTEM_B)
res_abs_b = run_drafts(abstain_dev, system=SYSTEM_B)
print("prompt B, answerable dev:"); summarize(res_dev_b)
print("prompt B, abstain dev:  "); summarize(res_abs_b)

prompt B, answerable dev:
n=46 | refused 10 (22%) | of answered: with citation 36/36, cites correct document 33/36
prompt B, abstain dev:  
n=7 | refused 7 (100%) | of answered: with citation 0/0, cites correct document 0/0


In [52]:
for r, o in res_dev_b:
    if not o["abstained"] and not any(o["passages"][i - 1][0] in gold[r["intent"]] for i in o["cited"]):
        print(r["intent"], "|", r["query"])
        print("   cited docs:", sorted({o["passages"][i - 1][0] for i in o["cited"]}))
        print("   draft:", o["draft"][:200], "\n")

change_shipping_address | I want to know about correcting the address
   cited docs: ['invoices']
   draft: If the name or address on an invoice is wrong, you can contact customer support on +91 00000 00000 or support@shopease.example (Monday to Friday, 9 AM to 6 PM) with your invoice number [4]. A correcte 

delivery_options | is it possible to place an order from Delhi
   cited docs: ['place-order', 'shipping-address']
   draft: Yes, it is possible to place an order from Delhi, as we deliver within India [1]. To place an order, you need a ShopEase account, which you can create for free on www.shopease.example [3]. 

delivery_period | I don't know how to see how soon ca I expect my article
   cited docs: ['track-order']
   draft: The estimated delivery date is shown on the order page and in your dispatch email [1]. When your order is dispatched, you will receive an email with a tracking number starting with TRK [2]. Please not 



In [53]:
res_dev_nf = [(r, draft(gem, chunked, r["query"], floor=0.0, system=SYSTEM_B)) for r in dev_all]
res_abs_nf = [(r, draft(gem, chunked, r["query"], floor=0.0, system=SYSTEM_B)) for r in abstain_dev]
summarize(res_dev_nf); summarize(res_abs_nf)

n=46 | refused 9 (20%) | of answered: with citation 37/37, cites correct document 34/37
n=7 | refused 7 (100%) | of answered: with citation 0/0, cites correct document 0/0


In [54]:
offtopic = ["What is the capital of France?", "Write me a poem about the sea",
            "Can you recommend a good laptop?", "How do I reset my home wifi router?",
            "Do you sell gift cards?"]
for q in offtopic:
    o = draft(gem, chunked, q, floor=0.0, system=SYSTEM_B)
    print(q, "->", "REFUSED" if o["abstained"] else o["draft"][:100], "| top score:", round(o["passages"][0][2], 3))

What is the capital of France? -> REFUSED | top score: 0.381
Write me a poem about the sea -> REFUSED | top score: 0.41
Can you recommend a good laptop? -> REFUSED | top score: 0.439
How do I reset my home wifi router? -> REFUSED | top score: 0.595
Do you sell gift cards? -> REFUSED | top score: 0.635


In [55]:
SYSTEM_C = SYSTEM_B + (" 5. If the message is too vague, or could mean more than one thing covered by the passages, "
                       "ask ONE short clarifying question instead of answering or refusing. A question needs no citation.")

def outcome(o):
    if o["abstained"]:
        return "refused"
    if not o["cited"] and "?" in o["draft"]:
        return "clarify"
    return "answered"

from collections import Counter
res_dev_c = run_drafts(dev_all, system=SYSTEM_C)
res_abs_c = run_drafts(abstain_dev, system=SYSTEM_C)
print("answerable:", Counter(outcome(o) for _, o in res_dev_c))
print("abstain:   ", Counter(outcome(o) for _, o in res_abs_c))

answerable: Counter({'answered': 34, 'refused': 9, 'clarify': 3})
abstain:    Counter({'refused': 7})


In [56]:
import importlib, drafting
importlib.reload(drafting)
from drafting import draft

res_dev_b = run_drafts(dev_all, system=SYSTEM_B)
res_abs_b = run_drafts(abstain_dev, system=SYSTEM_B)
res_dev_c = run_drafts(dev_all, system=SYSTEM_C)
res_abs_c = run_drafts(abstain_dev, system=SYSTEM_C)

print("B answerable:", Counter(outcome(o) for _, o in res_dev_b), "| abstain:", Counter(outcome(o) for _, o in res_abs_b))
print("C answerable:", Counter(outcome(o) for _, o in res_dev_c), "| abstain:", Counter(outcome(o) for _, o in res_abs_c))

B answerable: Counter({'answered': 37, 'refused': 9}) | abstain: Counter({'refused': 7})
C answerable: Counter({'answered': 35, 'refused': 8, 'clarify': 3}) | abstain: Counter({'refused': 7})


In [57]:
b_by_id = {r["ticket_id"]: o for r, o in res_dev_b}
for r, o in res_dev_c:
    if outcome(o) == "clarify":
        print(r["intent"], "|", r["query"])
        print("   C:", o["draft"])
        print("   B:", ("(refused)" if b_by_id[r["ticket_id"]]["abstained"] else b_by_id[r["ticket_id"]]["draft"][:150]), "\n")

change_shipping_address | I want to know about correcting the address
   C: Could you please clarify whether you need to change the delivery address on an existing order, save a new address to your account, or correct the address on an invoice?
   B: If the name or address on an invoice is wrong, you can contact customer support on +91 00000 00000 or support@shopease.example (Monday to Friday, 9 AM 

change_shipping_address | want assistance to update the address
   C: Would you like to update your saved delivery addresses in your account, change the address on an existing order, or correct the address on an invoice?
   B: To update the delivery address on an existing order that has not been dispatched, open **My Orders**, select the order, click **Change Order**, and up 

payment_issue | I need assistance to report troubles with online payment
   C: Are you having trouble with a payment failing, or do you see a duplicate or unauthorized charge?
   B: Hello! If your online payment fail

In [58]:
for r, o in res_dev_c:
    if outcome(o) == "refused":
        print(r["intent"], "|", r["query"], "|", o["reason"])

check_invoice | could I see the billfrom Sara Brown? | model_judgement
check_refund_policy | I need to see in which cases can I ask for a reimbursement | model_judgement
create_account | where to create a Premium account? | model_judgement
delete_account | clsoe Basic account | model_judgement
place_order | assistance shopping a few of ur product | model_judgement
place_order | earning product | model_judgement
recover_password | I want assistance to reset the PIN code of my account | model_judgement
track_refund | I expect a restitution of 999 dollars | model_judgement


In [60]:
import pandas as pd
from pathlib import Path
for name in ["past", "dev", "test", "abstain"]:
    d = pd.read_csv(Path("data/processed") / f"{name}_clean.csv", keep_default_na=False, encoding="utf-8")
    text = d["instruction"] + " " + d["response"]
    print(name, "| braces:", text.str.contains(r"[{}]").sum(),
          "| 'Account Type' or 'Account Category':", text.str.contains("Account Type|Account Category").sum())

past | braces: 0 | 'Account Type' or 'Account Category': 1
dev | braces: 0 | 'Account Type' or 'Account Category': 0
test | braces: 0 | 'Account Type' or 'Account Category': 0
abstain | braces: 0 | 'Account Type' or 'Account Category': 0


In [62]:
import importlib, verification
importlib.reload(verification)
from verification import verify

gem_v = LLM(model="gemini-3.5-flash", min_interval=6.5)    # set min_interval from this model's own RPM

In [63]:
from drafting import draft
import drafting

for r in dev10[:4]:
    out = draft(gem, chunked, r["query"], system=SYSTEM_C)
    print(r["intent"], "|", r["query"])
    for v in verify(gem_v, out, r["query"]):
        print(f'   [{v["llm_label"]:11s}] {v["sentence"][:90]}  {v["bad_tokens"] or ""}')

cancel_order | I don't want purchase ORD-87590
   [not_factual] Hello!  
   [supported  ] You can cancel an order at any time before it is dispatched [2].  
   [supported  ] To cancel online, sign in to the ShopEase customer portal, open **My Orders**, select the   
   [supported  ] If you cannot cancel online, contact customer support on +91 00000 00000 or support@shopea  
cancel_order | want help canceling purchase ORD-20607
   [not_factual] Hello!  
   [supported  ] You can cancel your order at any time before it is dispatched [2].  
   [supported  ] To cancel, sign in to the ShopEase customer portal and open **My Orders**, select your ord  
   [supported  ] If you cannot cancel online, you can contact customer support at +91 00000 00000 or suppor  
   [supported  ] If the order is already dispatched, cancellation is no longer possible, but you can refuse  
change_order | modifying purchase ORD-76074
   [supported  ] You can change an order until it is dispatched [4].  
   [supporte